# Student Dropout & Academic Success — Model Training

**Dataset:** UCI *Predict Students' Dropout and Academic Success* (id 697)
**Task:** multi-class classification (Dropout / Enrolled / Graduate)

This notebook:
1. explores and cleans the data,
2. creates a stratified 80/20 train/test split (test split saved as `../test_data.csv`),
3. trains 5 classifiers, each as a full scikit-learn `Pipeline` (preprocessing + model),
4. evaluates them with the shared `utils.evaluate()` function (same code the Streamlit app uses),
5. saves the pipelines + metadata into this `model/` folder.

Run it from the `model/` folder (Jupyter does this automatically when you open the notebook here).

In [ ]:
# ---- Imports -------------------------------------------------------------
import sys, json, time, platform
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# ---- Paths -----------------------------------------------------------------
# The notebook lives in model/, so the project root is one level up.
MODEL_DIR = Path.cwd() if Path.cwd().name == 'model' else Path.cwd() / 'model'
ROOT = MODEL_DIR.parent

# Make utils.py (in the project root) importable from this notebook
sys.path.insert(0, str(ROOT))
from utils import evaluate, clean_columns, METRIC_NAMES

RANDOM_STATE = 42          # fixed seed -> reproducible split and models
DATA_FILE = ROOT / 'data' / 'students_dropout.csv'
TARGET = 'Target'
print('Project root:', ROOT)

## 1. Load and explore the data

In [ ]:
# The UCI file is semicolon-separated and starts with a BOM -> use sep=';' and utf-8-sig
df = pd.read_csv(DATA_FILE, sep=';', encoding='utf-8-sig')
df = clean_columns(df)     # removes the stray tab in 'Daytime/evening attendance'

print('Shape (rows, columns):', df.shape)
print('Number of input features:', df.shape[1] - 1)
df.head()

In [ ]:
# Data types: all features are numeric (categorical ones are already integer-coded by UCI)
print(df.dtypes.value_counts(), '\n')
df.info()

In [ ]:
# Missing values and duplicates
missing = df.isna().sum()
print('Columns with missing values:')
print(missing[missing > 0] if missing.any() else '  none')
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
# Class balance: Enrolled is the minority class, so we use a STRATIFIED split
# and weighted-average metrics later.
counts = df[TARGET].value_counts()
balance = pd.DataFrame({'count': counts, 'percent': (counts / len(df) * 100).round(2)})
balance

In [ ]:
# Quick summary statistics of the numeric features
df.describe().T.round(2)

## 2. Handle missing values and encode the target

* Rows without a target label are dropped (we cannot learn from them).
* Missing **feature** values are imputed *inside* the pipeline (median for numeric, most frequent for categorical).
  This way exactly the same rule is applied to new data in the Streamlit app.
* The text target is mapped to integers `0..n-1`; the mapping is stored in `metadata.json`.

In [ ]:
df = df.dropna(subset=[TARGET]).reset_index(drop=True)

# Alphabetical order -> Dropout=0, Enrolled=1, Graduate=2
class_names = sorted(df[TARGET].astype(str).unique())
label_mapping = {name: idx for idx, name in enumerate(class_names)}
print('Label mapping:', label_mapping)

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(str).map(label_mapping)

# Split the feature columns by type so each type gets the right preprocessing
numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]
print(f'{len(numeric_cols)} numeric columns, {len(categorical_cols)} categorical columns')

## 3. Stratified 80/20 train/test split
The test split is saved with the **original column names, unscaled, with the text target** so it can be uploaded straight into the Streamlit app.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

# Save the held-out test data (raw features + original text label)
test_df = X_test.copy()
test_df[TARGET] = df.loc[X_test.index, TARGET].values
test_df.to_csv(ROOT / 'test_data.csv', index=False)
print('Saved', ROOT / 'test_data.csv')

# Check that the class proportions are preserved in both splits
pd.DataFrame({'train %': y_train.value_counts(normalize=True).sort_index() * 100,
              'test %': y_test.value_counts(normalize=True).sort_index() * 100},
             ).rename(index=dict(enumerate(class_names))).round(2)

## 4. Preprocessing (ColumnTransformer)
* Numeric columns: median imputation, then `StandardScaler` (important for distance/margin based models: KNN, SVM, Logistic Regression).
* Categorical (text) columns: most-frequent imputation, then `OneHotEncoder` — only added if such columns exist (this dataset has none).

In [ ]:
def make_one_hot_encoder():
    """OneHotEncoder with dense output that works on old and new scikit-learn.
    sklearn >= 1.2 calls the argument 'sparse_output', older versions call it 'sparse'."""
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def build_preprocessor():
    """Return a NEW (unfitted) preprocessor; every model pipeline gets its own copy."""
    transformers = [
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), numeric_cols)
    ]
    if categorical_cols:
        transformers.append(
            ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                              ('onehot', make_one_hot_encoder())]), categorical_cols)
        )
    return ColumnTransformer(transformers)

build_preprocessor()

## 5. Define and train the 5 models
Each model is wrapped in a `Pipeline(preprocessing -> model)`, so the saved file contains *everything* needed to go from raw CSV values to a prediction.

In [ ]:
# file name stem -> (display name, estimator)
models = {
    'logistic_regression': ('Logistic Regression',
                            LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    # max_depth limited so the tree does not memorise the training data
    'decision_tree': ('Decision Tree',
                      DecisionTreeClassifier(max_depth=6, min_samples_leaf=10,
                                             random_state=RANDOM_STATE)),
    # k=15 smooths the decision boundary compared to the default k=5
    'knn': ('KNN', KNeighborsClassifier(n_neighbors=15)),
    'naive_bayes': ('Naive Bayes', GaussianNB()),
    # probability=True is needed for predict_proba (AUC); RBF kernel = non-linear boundary
    'svm': ('Support Vector Machine',
            SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)),
}

pipelines, results, fit_report = {}, [], []

for key, (name, estimator) in models.items():
    pipe = Pipeline([('preprocess', build_preprocessor()), ('model', estimator)])

    start = time.time()
    pipe.fit(X_train, y_train)                     # learn ONLY from training data
    seconds = time.time() - start

    metrics, _, _ = evaluate(pipe, X_test, y_test) # score on unseen test data
    train_acc = accuracy_score(y_train, pipe.predict(X_train))

    pipelines[key] = pipe
    results.append({'ML Model Name': name, **metrics})
    fit_report.append({'ML Model Name': name, 'Train Accuracy': train_acc,
                       'Test Accuracy': metrics['Accuracy'],
                       'Gap (train - test)': train_acc - metrics['Accuracy'],
                       'Fit time (s)': seconds})
    print(f'{name:<24} trained in {seconds:5.1f}s  test accuracy = {metrics["Accuracy"]:.4f}')

## 6–7. Comparison table

In [ ]:
results_df = pd.DataFrame(results).set_index('ML Model Name')[METRIC_NAMES].round(4)
results_df.to_csv(MODEL_DIR / 'metrics_comparison.csv')
print('Saved', MODEL_DIR / 'metrics_comparison.csv')
results_df

In [ ]:
# Same table as Markdown -> copy-paste into README.md
def to_markdown_table(frame):
    header = '| ' + ' | '.join([frame.index.name] + list(frame.columns)) + ' |'
    divider = '|' + '---|' * (len(frame.columns) + 1)
    rows = ['| ' + ' | '.join([str(idx)] + [f'{v:.4f}' for v in row]) + ' |'
            for idx, row in frame.iterrows()]
    return '\n'.join([header, divider] + rows)

print(to_markdown_table(results_df))

In [ ]:
# Train vs test accuracy: a large gap means overfitting
pd.DataFrame(fit_report).set_index('ML Model Name').round(4)

In [ ]:
# Per-class report for each model (useful when writing the observations)
for key, (name, _) in models.items():
    print('=' * 60, '\n', name)
    print(classification_report(y_test, pipelines[key].predict(X_test),
                                target_names=class_names, digits=4, zero_division=0))

## 8. Save the pipelines and metadata

In [ ]:
model_files = {}
for key, (name, _) in models.items():
    path = MODEL_DIR / f'{key}.joblib'
    joblib.dump(pipelines[key], path)
    model_files[name] = path.name
    print(f'Saved {path.name:<28} ({path.stat().st_size / 1024:.0f} KB)')

# Everything the Streamlit app needs to validate and encode an uploaded CSV
metadata = {
    'dataset': "UCI Predict Students' Dropout and Academic Success",
    'target': TARGET,
    'feature_columns': X.columns.tolist(),
    'numeric_columns': numeric_cols,
    'categorical_columns': categorical_cols,
    'class_names': class_names,
    'label_mapping': label_mapping,
    'model_files': model_files,
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'random_state': RANDOM_STATE,
    'sklearn_version': sklearn.__version__,
}
with open(MODEL_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved metadata.json')

## 9. Library versions
Pin **scikit-learn** and **numpy** in `requirements.txt` to these versions so Streamlit Cloud can load the saved `.joblib` files.

In [ ]:
print('Python      ', platform.python_version())
print('scikit-learn', sklearn.__version__)
print('numpy       ', np.__version__)
print('pandas      ', pd.__version__)
print('joblib      ', joblib.__version__)